# 04 - Baseline B3: árbol y bosque aleatorio

Se eligió **una configuración común de bosque** mediante el macro-AUC simple de `val`, dando el mismo peso a A, B y C. Los ajustes de esa configuración usan todo el `train` propio de cada escenario para evaluar tres protocolos distintos. La aplicación usará un solo modelo; la elección del archivo se hará según el uso previsto. La búsqueda y la evaluación de finalistas ya están cerradas; no se reentrena por defecto.


In [ ]:
# Cada cuaderno prepara su propio runtime de Colab; no depende de 00_arranque ni de otra sesión.
from pathlib import Path
import sys, json, time, subprocess
from datetime import datetime, timezone
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo fetch origin 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo checkout 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo pull --ff-only origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / 'src') not in sys.path: sys.path.insert(0, str(REPO_DIR / 'src'))
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp.ingest import load_days
from ptcg_wp.features import feature_columns
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b3 import fit_b3_candidates_val
DATA_DIR = Path('/content/datos')
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(p.exists() for p in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(p.exists() for p in esperados), 'Faltan tablas procesadas; revisa Kaggle.'
schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62
SEED = 2026
SCENARIOS = ('A', 'B', 'C')
DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
RUN_INFO = {'commit': COMMIT, 'working_tree_dirty': DIRTY, 'seed': SEED, 'run_id_utc': RUN_ID}
RESULTS_DIR = REPO_DIR / 'results' / 'baseline3'
print('Tablas listas:', len(list(DATA_DIR.glob('estados_*.parquet'))), 'días | commit:', COMMIT,
      '| cambios locales:', DIRTY, '| corrida UTC:', RUN_ID)


In [ ]:
for escenario in ('A', 'B', 'C'):
    ruta = RESULTS_DIR / f'b3_val_{escenario}.csv'
    print(f'Escenario {escenario}')
    if ruta.exists():
        datos = pd.read_csv(ruta)
        display(datos.round(4))
    else:
        print('No está disponible:', ruta)


## Artefactos y límites

Los CSV de validación documentan los ajustes con todo `train` de cada escenario. La celda de verificación comprueba los metadatos de los `.joblib` disponibles. No se repite la búsqueda ni se usa `test`. La selección se basó en valores observados, sin bootstrap emparejado ni afirmación de significancia estadística.


## Verificar y exportar modelos B3

Busca primero el ajuste local de la configuración seleccionada para cada escenario; si falta, busca en Drive. Verifica hiperparámetros, semilla y uso de todo `train`, y copia los archivos a `export_repo/baseline3`. No ajusta modelos.


In [ ]:
from pathlib import Path
import sys, shutil, hashlib, joblib
from ptcg_wp.finalist_evidence import selected_train_count, verify_full_train

def sha256_file(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
BUSCAR_MODELOS_B3 = False
if BUSCAR_MODELOS_B3:
    dest = root / 'export_repo' / 'baseline3'; dest.mkdir(parents=True, exist_ok=True)
    esperado = {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 20,
                'n_estimators': 25, 'max_features': 0.5}
    for escenario in ('A', 'B', 'C'):
        nombre = f'b3_{escenario}_B3_bosque_log_loss.joblib'
        local = RESULTS_DIR / 'evaluacion' / f'escenario_{escenario}' / nombre
        encontrados = [local] if local.exists() else list((root / 'b3').rglob(nombre))
        if len(encontrados) != 1:
            print(nombre, ': se esperaba 1 archivo; encontrados:', encontrados); continue
        src = encontrados[0]; item = joblib.load(src)
        assert item.get('escenario') == escenario and list(item['features']) == list(FEATURES)
        assert item.get('seleccionada_por') == 'log_loss' and int(item.get('trial')) == 1
        expected = selected_train_count(RESULTS_DIR / f'b3_val_{escenario}.csv', escenario, 'B3')
        verify_full_train(item, escenario, 'B3', expected)
        parametros = item['model'].get_params()
        assert all(parametros[k] == v for k, v in esperado.items()), parametros
        assert parametros['bootstrap'] is False and parametros['random_state'] == 2030
        assert item.get('version_sklearn') == '1.7.2'
        target = dest / nombre
        if target.exists():
            if sha256_file(target) != sha256_file(src):
                print('La copia existente difiere; no se sobrescribe:', target); continue
            print('Copia existente verificada:', target)
        else:
            shutil.copy2(src, target)
            print('Copiado:', target)
        print('B3', escenario, '| train completo:', expected, 'partidas | SHA256:', sha256_file(target),
              '| semilla del modelo:', parametros['random_state'], '| sklearn:', item.get('version_sklearn'))


## Generar solo el modelo B3 de evaluación que falte

Ejecuta primero la verificación. Si falta el `.joblib` seleccionado en el repositorio y en Drive, reajusta únicamente el bosque finalista con todo el `train` propio de ese escenario; luego evalúa `val` y guarda el artefacto. Las métricas de CV requeridas se recuperan de los cinco pliegues registrados. No corre la búsqueda ni ajusta otras candidatas.


In [ ]:
import time, json
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b3 import fit_b3_candidates_val
from ptcg_wp.finalist_evidence import b3_selected_candidate
GENERAR_B3_FINALISTA_SI_FALTA = False
if GENERAR_B3_FINALISTA_SI_FALTA:
    drive_root = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
    parametros = {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 20,
                  'n_estimators': 25, 'max_features': 0.5}
    for escenario in ('A', 'B', 'C'):
        nombre = f'b3_{escenario}_B3_bosque_log_loss.joblib'
        local = RESULTS_DIR / 'evaluacion' / f'escenario_{escenario}' / nombre
        existentes = [local] if local.exists() else list((drive_root / 'b3').rglob(nombre))
        if existentes:
            print(f'{escenario}: ya existe el artefacto; no se entrena:', existentes)
            continue
        salida = drive_root / 'b3' / 'finalista_seleccionada_refit' / RUN_ID / f'escenario_{escenario}'
        salida.mkdir(parents=True, exist_ok=False)
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES))
        # Reconstruye las métricas de los 5 pliegues CV: fit_b3_candidates_val las requiere.
        candidata = pd.DataFrame([b3_selected_candidate(
            RESULTS_DIR / f'b3_cv_{escenario}.csv', escenario)])
        inicio = time.perf_counter()
        metricas = fit_b3_candidates_val(stream, FEATURES, candidata,
            max_train_games=None, seed=2030, n_jobs=2, max_ram_gib=10.0,
            cache_dir='/content', output_dir=salida, verbose=True)
        registro = {**RUN_INFO, 'escenario': escenario, 'baseline': 'B3',
                    'configuracion': parametros, 'semilla_base_original': 2026,
                    'semilla_efectiva_modelo': 2030, 'segundos': time.perf_counter()-inicio}
        (salida / 'registro.json').write_text(json.dumps(registro, indent=2))
        display(metricas)
        print('Modelo guardado:', salida / nombre)


## Descargar resultados B3

Empaqueta los modelos finales exportados y, si se generaron en esta corrida, los manifiestos y métricas del reajuste finalista.
Ejecuta esta celda después de terminar las secciones que quieras conservar.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import shutil

def agregar_a_zip(archivo_zip, ruta, raiz_drive, carpeta_zip):
    ruta = Path(ruta)
    if not ruta.exists():
        print('No existe, se omite:', ruta)
        return 0
    n = 0
    for item in ruta.rglob('*'):
        if item.is_file():
            destino = Path(carpeta_zip) / item.relative_to(raiz_drive)
            archivo_zip.write(item, destino.as_posix(), compress_type=ZIP_DEFLATED, compresslevel=1)
            n += 1
    return n
raiz = DRIVE_ROOT.parent
zip_path = Path('/content') / f'resultados_B3_{RUN_ID}.zip'
with ZipFile(zip_path, 'w') as zf:
    cantidad = 0
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'export_repo' / 'baseline3', raiz, 'Applied-AI-Agent-Project')
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'b3' / 'finalista_seleccionada_refit' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
assert cantidad > 0, 'No hay modelos B3 exportados; ejecuta la recuperación o el reajuste de los finalistas.'
print('Archivos:', cantidad, '| ZIP:', zip_path, '| tamaño MiB:', round(zip_path.stat().st_size / 2**20, 1))
files.download(str(zip_path))
